# Reproducing LongMemEval-V2's baseline results

**Question this notebook answers:** on LongMemEval-V2 (Wu, Ji, Kawatkar, Kwan, Gu, Peng, Chang -- UCLA,
arXiv:2605.12493v1), do the paper's *baseline* memory methods -- no retrieval, RAG over raw trajectory-state
slices, and RAG augmented with LLM-generated notes -- reproduce the paper's reported Small-tier scores of
**1.3% / 42.8% / 51.0%** (Table 2)?

**Scope, deliberately:** this notebook reproduces three of Table 2's six rows -- `no_retrieval`,
`rag_query_to_slice`, `rag_query_to_slice_notes` -- on the **Small** tier, across both domains (`web`,
`enterprise`). It does **not** run `codex`, `agentrunbook_r`, or `agentrunbook_c` -- the first is a much
more expensive off-the-shelf-agent baseline, and the latter two are the paper's own proposed methods, not
baselines. See `.claude/plans/c-users-amjad-downloads-longmemeval-v2-replicated-rocket.md` for the full
reasoning behind this scope and its cost/time estimates.

**Method, in one paragraph:** rather than re-implement the benchmark from the paper's description -- which
would guarantee *not* matching its numbers, since prompt wording, context-truncation order, and scoring
details would all drift -- this notebook clones and drives the paper's own released, Apache-2.0 harness
(`github.com/xiaowu0162/LongMemEval-V2`) against its official public dataset
(`huggingface.co/datasets/xiaowu0162/longmemeval-v2`). The harness's hardcoded hyperparameters are used
unmodified throughout. The only configuration this notebook supplies is *where the models live* -- hosted
OpenAI-compatible endpoints for the reader (`Qwen/Qwen3.5-9B`) and embedder (`Qwen/Qwen3-Embedding-8B`),
since neither fits on a single consumer GPU at the paper's 200K-token context budget -- and OpenAI directly
for the judge (`gpt-5.2`, medium reasoning), which the harness always requires regardless of where the
reader is hosted.

**Gates, not a single pass/fail:** the notebook stops at checkpoints -- a cheap plumbing smoke test, the
no-context floor (the single most informative check: roughly 1.3%, or something upstream is leaking
answers), the three baseline scores against Table 2, and an optional seed-variance run -- so a problem is
caught before money is spent on the next stage, not after.


## 1. Environment


In [1]:
import sys
import platform

IN_COLAB = "google.colab" in sys.modules
IS_NATIVE_WINDOWS = platform.system() == "Windows" and not IN_COLAB
print("Running in Colab:", IN_COLAB)
print("Platform:", platform.system(), platform.release())

if IS_NATIVE_WINDOWS:
    print(
        "\nWARNING: running on native Windows. The harness's data-prep step "
        "(data/prepare_data.py) defaults to symlinking screenshot directories, "
        "which needs Developer Mode or admin on Windows. This notebook falls "
        "back to --mode copy automatically (Section 3), which works but uses "
        "more disk. For the smoothest run, use Colab or a Linux box instead -- "
        "see the plan file for why (git clone / editable install / pip install "
        "also all assume a POSIX-ish shell more comfortably than native Windows)."
    )


Running in Colab: False
Platform: Windows 11



### Install the harness's dependencies

The official README installs via `conda env create -f environment.yml`, which pulls in
`requirements-torch.txt` (`torch==2.6.0+cu124`, etc.). **This notebook skips that file on purpose.** None of
the three baseline arms run here do local GPU inference -- the reader, controller, and embedder are all
hosted over HTTP -- so torch isn't needed, and installing it risks clobbering the CUDA-matched torch build
this repo's shared `.venv` already has installed for the other notebooks
(`fine_tune_qwen_gsm8k_lora.ipynb` etc.). `pyproject.toml` confirms torch is only an *optional* extra
(`[project.optional-dependencies].torch`), not a core dependency, so `pip install -e .` alone is safe.


In [2]:
# The harness's own pinned dependencies (requirements.txt), minus torch -- see the
# note above for why torch is skipped. `pandas` is not part of the harness; it's
# used only by this notebook's own comparison tables further down. `python-dotenv`
# is also not part of the harness -- it's used only to load API keys from the
# repo-root .env file in the "API keys" section below.
%pip install -q huggingface_hub numpy openai openai-agents pillow tqdm transformers pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


### Clone and install the harness


In [3]:
import subprocess
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_ROOT = Path("/content/drive/MyDrive/algoverse_longmemeval_v2")
else:
    # Mirrors this repo's existing convention (see fine_tune_qwen_gsm8k_lora.ipynb):
    # relative paths under the notebook's own working directory, kept separate from
    # the repo's own tracked files since this pulls down several GB of harness + data.
    WORK_ROOT = Path.cwd() / "lme_v2_work"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
HARNESS_DIR = WORK_ROOT / "LongMemEval-V2"
DATA_ROOT = WORK_ROOT / "data" / "longmemeval-v2"
OUTPUT_ROOT = WORK_ROOT / "runs"
print("WORK_ROOT:", WORK_ROOT)

if not HARNESS_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/xiaowu0162/LongMemEval-V2.git", str(HARNESS_DIR)],
        check=True,
    )
else:
    print("Harness already cloned at", HARNESS_DIR)

# Editable install of the harness package (data/, evaluation/, memory_modules/).
# Safe re: torch -- see the note above; pyproject.toml lists torch only under
# [project.optional-dependencies], not as a core dependency, so this won't touch it.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(HARNESS_DIR)], check=True)
print("Harness installed.")


WORK_ROOT: D:\Projects\Algoverse\lme_v2_work
Harness already cloned at D:\Projects\Algoverse\lme_v2_work\LongMemEval-V2


Harness installed.


## 2. Configuration

Model identities, tier, and output locations. The model names are pinned by the paper -- change *where*
they're served (`*_BASE_URL`) freely, but not `READER_MODEL` / `EMBEDDING_MODEL` / `EVALUATOR_MODEL`
themselves.


In [4]:
import os

TIER = "small"                     # this notebook's scope -- see the plan file for the Medium-tier caveats
DOMAINS = ["web", "enterprise"]
ARMS = ["no_retrieval", "rag_query_to_slice", "rag_query_to_slice_notes"]

# --- Reader (answers the questions) ---
READER_MODEL = os.environ.setdefault("READER_MODEL", "Qwen/Qwen3.5-9B")
READER_BASE_URL = os.environ.setdefault("READER_BASE_URL", "https://openrouter.ai/api/v1")
READER_API_KEY_ENV = "OPENROUTER_API_KEY"  # must be set in the notebook's environment

# --- Controller (generates procedure/hint notes at insert time; RAG arms only) ---
CONTROLLER_MODEL = os.environ.setdefault("LME_CONTROLLER_MODEL", READER_MODEL)
CONTROLLER_BASE_URL = os.environ.setdefault("LME_CONTROLLER_BASE_URL", READER_BASE_URL)
CONTROLLER_API_KEY_ENV = "OPENROUTER_API_KEY"

# --- Embedder (retrieval; RAG arms only) ---
EMBEDDING_MODEL = os.environ.setdefault("LME_EMBEDDING_MODEL", "Qwen/Qwen3-Embedding-8B")
EMBEDDING_BASE_URL = os.environ.setdefault("LME_EMBEDDING_BASE_URL", "https://openrouter.ai/api/v1")
EMBEDDING_API_KEY_ENV = "OPENROUTER_API_KEY"

# --- Judge (scores gotchas + abstention questions; always OpenAI, never swappable) ---
EVALUATOR_MODEL = "gpt-5.2"
EVALUATOR_API_KEY_ENV = "OPENAI_API_KEY"

# evaluation/run_eval.py's --reader-api-key-env / --controller-api-key-env /
# --embedding-api-key-env each independently *default* to the literal string
# "OPENAI_API_KEY" -- which would misroute the OpenRouter key to api.openai.com
# if left unset. run_lme() below passes all three explicitly for this reason;
# do not rely on the CLI's own defaults.

print(f"Reader/controller/embedder -> {READER_BASE_URL}")
print(f"Judge -> OpenAI ({EVALUATOR_MODEL})")

Reader/controller/embedder -> https://openrouter.ai/api/v1
Judge -> OpenAI (gpt-5.2)


### API keys

Loaded from a local `.env` file at the repo root (`D:\Projects\Algoverse\.env`, never committed/printed)
when running locally -- never hardcoded in the notebook. Colab secrets (the key icon in the left sidebar)
are preferred when running in Colab; if neither a `.env` entry nor a Colab secret is found, this falls
back to a masked local prompt.

In [5]:
import getpass
from dotenv import find_dotenv, load_dotenv

# Loads Algoverse/.env (found by walking up from cwd) into os.environ, without
# overriding any var already set in the process environment -- so an explicit
# shell/Colab env var still wins over the file. Quietly no-ops if no .env is
# found (e.g. Colab), falling through to the userdata/prompt path below.
_dotenv_path = find_dotenv(usecwd=True)
if _dotenv_path:
    load_dotenv(_dotenv_path)
    print(f".env loaded from {_dotenv_path}")
else:
    print("No .env file found -- falling back to Colab userdata / env vars / prompt below.")

def get_secret(env_name: str) -> str:
    # Fetch a secret from an already-set env var (this includes anything load_dotenv
    # just populated above), Colab userdata, or a masked prompt -- in that order --
    # and stash it back into os.environ under env_name so the subprocess calls this
    # notebook makes to run_eval.py inherit it.
    if os.environ.get(env_name):
        return os.environ[env_name]
    if IN_COLAB:
        try:
            from google.colab import userdata
            value = userdata.get(env_name)
            if value:
                os.environ[env_name] = value
                return value
        except Exception:
            pass
    value = getpass.getpass(f"Enter {env_name}: ")
    if not value:
        raise RuntimeError(f"{env_name} is required and was not provided.")
    os.environ[env_name] = value
    return value

get_secret("OPENROUTER_API_KEY")
get_secret("OPENAI_API_KEY")
print("Keys loaded into the environment (not printed, not persisted in this notebook's output).")

.env loaded from D:\Projects\Algoverse\.env
Keys loaded into the environment (not printed, not persisted in this notebook's output).


## 3. Download and prepare the dataset

~7.1 GB from Hugging Face, then screenshots are materialized into the runtime layout the harness expects.
`--check-screenshots` is left on (the default) -- gotchas questions can depend on screenshot-visible
evidence in the memory context, so skipping screenshot prep would silently degrade that category rather
than fail loudly.


In [6]:
import time

DATA_PREP_MODE = "copy" if IS_NATIVE_WINDOWS else "symlink"
print(f"Data-prep link mode: {DATA_PREP_MODE}")

def run_data_script(script_name: str, *args: str, retries: int = 5) -> None:
    # The HF Hub download in particular is prone to transient connection resets
    # (WinError 10054 / RemoteProtocolError) on some networks. huggingface_hub's
    # snapshot_download writes straight into --data-root and skips files already
    # completed on disk, so retrying is a genuine resume, not a restart -- this
    # is purely network resilience and has no bearing on what gets downloaded or
    # how the harness itself runs.
    cmd = [sys.executable, str(HARNESS_DIR / "data" / script_name), *args]
    print("Running:", " ".join(cmd))
    last_exc: Exception | None = None
    for attempt in range(1, retries + 1):
        try:
            subprocess.run(cmd, check=True, cwd=str(HARNESS_DIR))
            return
        except subprocess.CalledProcessError as exc:
            last_exc = exc
            if attempt < retries:
                wait = 5 * attempt
                print(f"{script_name} failed on attempt {attempt}/{retries} ({exc}); retrying in {wait}s...")
                time.sleep(wait)
    raise last_exc

run_data_script("download_data.py", "--data-root", str(DATA_ROOT))
run_data_script("prepare_data.py", "--data-root", str(DATA_ROOT), "--mode", DATA_PREP_MODE)
run_data_script("validate_data.py", "--data-root", str(DATA_ROOT), "--tier", TIER)

Data-prep link mode: copy
Running: D:\Projects\Algoverse\.venv\Scripts\python.exe D:\Projects\Algoverse\lme_v2_work\LongMemEval-V2\data\download_data.py --data-root D:\Projects\Algoverse\lme_v2_work\data\longmemeval-v2
Running: D:\Projects\Algoverse\.venv\Scripts\python.exe D:\Projects\Algoverse\lme_v2_work\LongMemEval-V2\data\prepare_data.py --data-root D:\Projects\Algoverse\lme_v2_work\data\longmemeval-v2 --mode copy


Running: D:\Projects\Algoverse\.venv\Scripts\python.exe D:\Projects\Algoverse\lme_v2_work\LongMemEval-V2\data\validate_data.py --data-root D:\Projects\Algoverse\lme_v2_work\data\longmemeval-v2 --tier small


## 4. Preflight: verify the hosted endpoints before spending on real runs

Three checks the harness's own code doesn't do for you, each targeting a specific way a hosted provider
can silently drift from what the paper assumes:

1. **The reader accepts `top_k` in `extra_body`.** `harness.py`'s `build_extra_body()` always sends
   `top_k: 20` (the paper's sampling setting) regardless of provider -- some OpenAI-compatible endpoints
   reject unknown sampling params outright.
2. **The embedder returns a real vector.** Confirms the model id resolves and the endpoint is reachable.
3. **The judge (OpenAI directly) authenticates and responds.** No `temperature` override -- reasoning-effort
   models reject it.

Fix any failure here before Section 6 -- it will otherwise surface hours into a real run instead of in
thirty seconds now.


In [7]:
from openai import OpenAI

def preflight_reader() -> None:
    client = OpenAI(base_url=READER_BASE_URL, api_key=os.environ[READER_API_KEY_ENV])
    resp = client.chat.completions.create(
        model=READER_MODEL,
        messages=[{"role": "user", "content": "Reply with exactly one word: OK"}],
        max_tokens=8,
        temperature=0.6,
        top_p=0.95,
        extra_body={"top_k": 20},
    )
    print("Reader OK -- reported model:", resp.model)
    print("Sample reply:", resp.choices[0].message.content)

def preflight_embedder() -> None:
    client = OpenAI(base_url=EMBEDDING_BASE_URL, api_key=os.environ[EMBEDDING_API_KEY_ENV])
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=["preflight check"])
    print("Embedder OK -- vector dimension:", len(resp.data[0].embedding))

def preflight_judge() -> None:
    client = OpenAI(api_key=os.environ[EVALUATOR_API_KEY_ENV])
    resp = client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[{"role": "user", "content": "Reply with exactly one word: OK"}],
        max_completion_tokens=16,
    )
    print("Judge OK -- sample reply:", resp.choices[0].message.content)

for name, fn in [("reader", preflight_reader), ("embedder", preflight_embedder), ("judge", preflight_judge)]:
    print(f"\n--- Preflighting {name} ---")
    try:
        fn()
    except Exception as exc:
        print(f"PREFLIGHT FAILED for {name}: {exc}")
        print("Fix this before continuing -- see Section 4's checklist above for likely causes.")



--- Preflighting reader ---


Reader OK -- reported model: qwen/qwen3.5-9b
Sample reply: None

--- Preflighting embedder ---


Embedder OK -- vector dimension: 4096

--- Preflighting judge ---


Judge OK -- sample reply: OK


## 5. Harness driver helpers

`run_eval.py`'s hyperparameters (temperature, top_p, top_k, max_completion_tokens,
memory_context_max_tokens, concurrency, thinking-enabled) are already hardcoded to match the paper --
`run_lme()` below passes only identity (method/domain/tier/output path), endpoints, and API-key routing,
and leaves everything else at its default. This mirrors what `evaluation/scripts/run_{method}.sh` does
(loop over both domains), reimplemented in Python so it works identically on Windows and Colab without a
bash dependency.


In [8]:
import json
import math
import shutil

def run_lme(
    method: str,
    domain: str,
    *,
    tier: str = TIER,
    limit: int | None = None,
    shuffle_seed: int | None = None,
    output_root: Path = OUTPUT_ROOT,
    subprocess_timeout: float | None = None,
    retries: int = 1,
) -> Path:
    assert method in ARMS, f"{method!r} is outside this notebook's scope: {ARMS}"
    out_dir = output_root / f"{method}_{domain}_{tier}"
    cmd = [
        sys.executable, str(HARNESS_DIR / "evaluation" / "run_eval.py"),
        "--method", method,
        "--data-root", str(DATA_ROOT),
        "--domain", domain,
        "--tier", tier,
        "--output-dir", str(out_dir),
        "--reader-model", READER_MODEL,
        "--reader-base-url", READER_BASE_URL,
        "--reader-api-key-env", READER_API_KEY_ENV,
        "--controller-model", CONTROLLER_MODEL,
        "--controller-base-url", CONTROLLER_BASE_URL,
        "--controller-api-key-env", CONTROLLER_API_KEY_ENV,
        "--embedding-model", EMBEDDING_MODEL,
        "--embedding-base-url", EMBEDDING_BASE_URL,
        "--embedding-api-key-env", EMBEDDING_API_KEY_ENV,
        "--evaluator-model", EVALUATOR_MODEL,
        "--evaluator-api-key-env", EVALUATOR_API_KEY_ENV,
        "--evaluator-reasoning-effort", "medium",
    ]
    if limit is not None:
        cmd += ["--limit", str(limit)]
    if shuffle_seed is not None:
        cmd += ["--shuffle-questions-seed", str(shuffle_seed)]
    print("Running:", " ".join(cmd))
    # Two independent failure modes have been observed on OpenRouter, neither with a
    # CLI-level fix: (1) a single reader/embedder request stalling well past any
    # reasonable duration, and (2) a request returning a 200 OK whose body has
    # `choices: null` instead of raising an HTTP error -- the harness's own error
    # handling (harness.py run_one()) only recovers from openai.BadRequestError, so
    # this crashes the whole subprocess with an unhandled TypeError partway through
    # generation. Retrying the whole subprocess is the only lever available without
    # touching harness code -- pure network resilience, not a change to any
    # hyperparameter the paper's numbers depend on.
    #
    # A retry after a crash (as opposed to a timeout) must wipe out_dir first:
    # harness.py hard-refuses to rebuild a memory workspace that already exists on
    # disk from the failed attempt (harness.py:1101, "Refusing to overwrite existing
    # memory workspace"), so leaving it in place would make every retry fail
    # instantly on that guard rather than actually retrying.
    for attempt in range(1, retries + 1):
        if attempt > 1 and out_dir.exists():
            shutil.rmtree(out_dir)
        try:
            subprocess.run(cmd, check=True, cwd=str(HARNESS_DIR), timeout=subprocess_timeout)
            return out_dir
        except subprocess.TimeoutExpired:
            if attempt >= retries:
                raise
            print(f"run_eval.py exceeded {subprocess_timeout}s on attempt {attempt}/{retries}; retrying...")
        except subprocess.CalledProcessError as exc:
            if attempt >= retries:
                raise
            print(f"run_eval.py exited {exc.returncode} on attempt {attempt}/{retries}; retrying...")
    return out_dir


def run_lme_both_domains(method: str, **kwargs) -> dict[str, Path]:
    return {domain: run_lme(method, domain, **kwargs) for domain in DOMAINS}


def combine_domains(method: str, tier: str = TIER, output_root: Path = OUTPUT_ROOT) -> dict:
    web_metrics = output_root / f"{method}_web_{tier}" / "aggregated_metrics.json"
    ent_metrics = output_root / f"{method}_enterprise_{tier}" / "aggregated_metrics.json"
    combined_path = output_root / f"{method}_{tier}_combined_metrics.json"
    cmd = [
        sys.executable, str(HARNESS_DIR / "leaderboard" / "combine_aggregated_metrics.py"),
        str(web_metrics), str(ent_metrics), "-o", str(combined_path),
    ]
    subprocess.run(cmd, check=True, cwd=str(HARNESS_DIR))
    return json.loads(combined_path.read_text(encoding="utf-8"))


def load_per_question(method: str, domain: str, tier: str = TIER, output_root: Path = OUTPUT_ROOT) -> list[dict]:
    path = output_root / f"{method}_{domain}_{tier}" / "per_question.jsonl"
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def wilson_interval(k: int, n: int, z: float = 1.96) -> tuple[float | None, float | None, float | None]:
    # 95% Wilson score interval for a binomial proportion -- appropriate here
    # because several of the paper's categories are small enough (gotchas:
    # n=29) that the normal approximation the paper itself avoids would give
    # a nonsensical/zero-width interval near 0 or 1.
    if n == 0:
        return (None, None, None)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = (z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denom
    return (p, max(0.0, center - half), min(1.0, center + half))

## 6. Gate 1 -- smoke test

Five questions, `no_retrieval` only, both domains. This should complete in well under a minute and confirms
the whole chain -- data paths, endpoint config, harness invocation, scoring -- works end to end before any
real spend.


In [ ]:
smoke_dirs = run_lme_both_domains("no_retrieval", limit=5, subprocess_timeout=600, retries=2)
for domain, out_dir in smoke_dirs.items():
    metrics = json.loads((out_dir / "aggregated_metrics.json").read_text(encoding="utf-8"))
    print(f"{domain}: {metrics['overall']['count_all_questions']} questions scored, "
          f"overall_full_set={metrics['overall']['overall_full_set']}")
print("\nSmoke test passed if both domains printed a score above without raising.")

Running: d:\Projects\Algoverse\.venv\Scripts\python.exe d:\Projects\Algoverse\lme_v2_work\LongMemEval-V2\evaluation\run_eval.py --method no_retrieval --data-root d:\Projects\Algoverse\lme_v2_work\data\longmemeval-v2 --domain web --tier small --output-dir d:\Projects\Algoverse\lme_v2_work\runs\no_retrieval_web_small --reader-model Qwen/Qwen3.5-9B --reader-base-url https://openrouter.ai/api/v1 --reader-api-key-env OPENROUTER_API_KEY --controller-model Qwen/Qwen3.5-9B --controller-base-url https://openrouter.ai/api/v1 --controller-api-key-env OPENROUTER_API_KEY --embedding-model Qwen/Qwen3-Embedding-8B --embedding-base-url https://openrouter.ai/api/v1 --embedding-api-key-env OPENROUTER_API_KEY --evaluator-model gpt-5.2 --evaluator-api-key-env OPENAI_API_KEY --evaluator-reasoning-effort medium --limit 5


## 7. Gate 2 -- calibration: the no-context floor

The paper reports **1.3%** here (Table 2, both tiers). This is the highest-value single check in this
notebook: it confirms the questions genuinely require memory and that goal text isn't leaking answers
through the prompt. **A floor much above ~2-3% means something is wrong upstream -- stop and debug rather
than spending on the RAG arms below.**


In [ ]:
# NOTE: this reuses the same output directories as the Section 6 smoke test (paths are
# keyed by method/domain/tier, not --limit), so the 5-question smoke results are
# overwritten here by the full 451-question run -- that's expected, not a bug.
no_retrieval_dirs = run_lme_both_domains("no_retrieval", subprocess_timeout=10800, retries=2)
no_retrieval_combined = combine_domains("no_retrieval")

floor_score = no_retrieval_combined["overall"]["overall_full_set"]
floor_n = no_retrieval_combined["overall"]["count_all_questions"]
print(f"no_retrieval combined score: {floor_score:.4f} over {floor_n} questions (paper: 0.013)")

if floor_score > 0.03:
    print(
        "\nWARNING: floor is more than double the paper's 1.3%. Likely causes: goal text "
        "leaking answer content into the prompt, a prompt-template mismatch, or the judge "
        "scoring UNKNOWN as correct somewhere. Inspect per_question.jsonl before proceeding."
    )
else:
    print("\nFloor is in the expected range -- proceeding to the RAG arms is reasonable.")


Running: D:\Projects\Algoverse\.venv\Scripts\python.exe D:\Projects\Algoverse\lme_v2_work\LongMemEval-V2\evaluation\run_eval.py --method no_retrieval --data-root D:\Projects\Algoverse\lme_v2_work\data\longmemeval-v2 --domain web --tier small --output-dir D:\Projects\Algoverse\lme_v2_work\runs\no_retrieval_web_small --reader-model Qwen/Qwen3.5-9B --reader-base-url https://openrouter.ai/api/v1 --reader-api-key-env OPENROUTER_API_KEY --controller-model Qwen/Qwen3.5-9B --controller-base-url https://openrouter.ai/api/v1 --controller-api-key-env OPENROUTER_API_KEY --embedding-model Qwen/Qwen3-Embedding-8B --embedding-base-url https://openrouter.ai/api/v1 --embedding-api-key-env OPENROUTER_API_KEY --evaluator-model gpt-5.2 --evaluator-api-key-env OPENAI_API_KEY --evaluator-reasoning-effort medium


Running: D:\Projects\Algoverse\.venv\Scripts\python.exe D:\Projects\Algoverse\lme_v2_work\LongMemEval-V2\evaluation\run_eval.py --method no_retrieval --data-root D:\Projects\Algoverse\lme_v2_work\data\longmemeval-v2 --domain enterprise --tier small --output-dir D:\Projects\Algoverse\lme_v2_work\runs\no_retrieval_enterprise_small --reader-model Qwen/Qwen3.5-9B --reader-base-url https://openrouter.ai/api/v1 --reader-api-key-env OPENROUTER_API_KEY --controller-model Qwen/Qwen3.5-9B --controller-base-url https://openrouter.ai/api/v1 --controller-api-key-env OPENROUTER_API_KEY --embedding-model Qwen/Qwen3-Embedding-8B --embedding-base-url https://openrouter.ai/api/v1 --embedding-api-key-env OPENROUTER_API_KEY --evaluator-model gpt-5.2 --evaluator-api-key-env OPENAI_API_KEY --evaluator-reasoning-effort medium


no_retrieval combined score: 0.0288 over 451 questions (paper: 0.013)

Floor is in the expected range -- proceeding to the RAG arms is reasonable.


## 8. Baseline arm: RAG -> slice

Embedding retrieval over raw trajectory-state slices (radius 1), top-6 per query. No notes, no controller
LLM calls at query time -- `enable_notes: false` in `rag_query_to_slice.json` means insertion only ever
builds and embeds slices. Paper: **0.428** (Small).

Two runs, not one: a `--limit 5` check first, into its own `OUTPUT_ROOT/quickcheck/` directory -- **not**
the real run's output dir. Unlike `no_retrieval` (Gates 1-2), which has no memory workspace and can safely
reuse one output directory across a smoke test and the full run, `harness.py` hard-refuses to rebuild a
memory workspace that already exists (`Refusing to overwrite existing memory workspace`, `harness.py:1101`)
-- so the two RAG-arm runs must write to separate directories, or the full run below crashes before a
single reader call. Insertion isn't gated by `--limit` -- the full 100-trajectory Small haystack gets
embedded either way -- so the quick check doesn't save embedding cost, but it does validate the embedding
path against real trajectory data (~$1-2) before the ~$3-4 reader spend across all 451 questions.


In [9]:
_ = run_lme_both_domains("rag_query_to_slice", limit=5, output_root=OUTPUT_ROOT / "quickcheck")
print("Quick check complete -- inspect the output above for errors before running the full arm below.")


Running: D:\Projects\Algoverse\.venv\Scripts\python.exe D:\Projects\Algoverse\lme_v2_work\LongMemEval-V2\evaluation\run_eval.py --method rag_query_to_slice --data-root D:\Projects\Algoverse\lme_v2_work\data\longmemeval-v2 --domain web --tier small --output-dir D:\Projects\Algoverse\lme_v2_work\runs\quickcheck\rag_query_to_slice_web_small --reader-model Qwen/Qwen3.5-9B --reader-base-url https://openrouter.ai/api/v1 --reader-api-key-env OPENROUTER_API_KEY --controller-model Qwen/Qwen3.5-9B --controller-base-url https://openrouter.ai/api/v1 --controller-api-key-env OPENROUTER_API_KEY --embedding-model Qwen/Qwen3-Embedding-8B --embedding-base-url https://openrouter.ai/api/v1 --embedding-api-key-env OPENROUTER_API_KEY --evaluator-model gpt-5.2 --evaluator-api-key-env OPENAI_API_KEY --evaluator-reasoning-effort medium --limit 5


Running: D:\Projects\Algoverse\.venv\Scripts\python.exe D:\Projects\Algoverse\lme_v2_work\LongMemEval-V2\evaluation\run_eval.py --method rag_query_to_slice --data-root D:\Projects\Algoverse\lme_v2_work\data\longmemeval-v2 --domain enterprise --tier small --output-dir D:\Projects\Algoverse\lme_v2_work\runs\quickcheck\rag_query_to_slice_enterprise_small --reader-model Qwen/Qwen3.5-9B --reader-base-url https://openrouter.ai/api/v1 --reader-api-key-env OPENROUTER_API_KEY --controller-model Qwen/Qwen3.5-9B --controller-base-url https://openrouter.ai/api/v1 --controller-api-key-env OPENROUTER_API_KEY --embedding-model Qwen/Qwen3-Embedding-8B --embedding-base-url https://openrouter.ai/api/v1 --embedding-api-key-env OPENROUTER_API_KEY --evaluator-model gpt-5.2 --evaluator-api-key-env OPENAI_API_KEY --evaluator-reasoning-effort medium --limit 5


Quick check complete -- inspect the output above for errors before running the full arm below.


In [ ]:
slice_dirs = run_lme_both_domains("rag_query_to_slice", subprocess_timeout=10800, retries=2)


Running: d:\Projects\Algoverse\.venv\Scripts\python.exe d:\Projects\Algoverse\lme_v2_work\LongMemEval-V2\evaluation\run_eval.py --method rag_query_to_slice --data-root d:\Projects\Algoverse\lme_v2_work\data\longmemeval-v2 --domain web --tier small --output-dir d:\Projects\Algoverse\lme_v2_work\runs\rag_query_to_slice_web_small --reader-model Qwen/Qwen3.5-9B --reader-base-url https://openrouter.ai/api/v1 --reader-api-key-env OPENROUTER_API_KEY --controller-model Qwen/Qwen3.5-9B --controller-base-url https://openrouter.ai/api/v1 --controller-api-key-env OPENROUTER_API_KEY --embedding-model Qwen/Qwen3-Embedding-8B --embedding-base-url https://openrouter.ai/api/v1 --embedding-api-key-env OPENROUTER_API_KEY --evaluator-model gpt-5.2 --evaluator-api-key-env OPENAI_API_KEY --evaluator-reasoning-effort medium
run_eval.py exited 1 on attempt 1/2; retrying...


NOTE (patched after the fact): the web-domain subprocess launched by this cell was killed mid-run by a WiFi drop and the cell above shows that failed attempt's stale CalledProcessError. The run was repaired out-of-band (same run_eval.py command, same output dir, executed outside the notebook kernel to avoid a file-write race with the concurrently-running Section 9 process) rather than re-executing this cell, since re-running here would have re-spent the already-good, untouched enterprise-domain result.

Confirmed final results (read directly from aggregated_metrics.json on disk):
  web:        overall_full_set = 0.3958  (n=240)
  enterprise: overall_full_set = 0.4502  (n=211)
  combined:   overall_full_set = 0.4213  (n=451)

slice_dirs is not referenced by any downstream cell, so this repair is purely a reporting fix; the notebook's data-driven Gate 3 comparison below reads the same files directly and is unaffected.

## 9. Baseline arm: RAG -> slice + notes

Same retrieval, plus two LLM-generated notes (procedure, hint) per trajectory, built once at insert time by
the controller model and searched alongside the slices at query time. Paper: **0.510** (Small) -- an
8.2-point gap over the plain-slice arm that isolates what the notes alone contribute.

Same two-run pattern as Section 8, for the same reason.


In [9]:
_ = run_lme_both_domains("rag_query_to_slice_notes", limit=5, output_root=OUTPUT_ROOT / "quickcheck", subprocess_timeout=10800, retries=2)
print("Quick check complete -- inspect the output above for errors before running the full arm below.")

Running: D:\Projects\Algoverse\.venv\Scripts\python.exe D:\Projects\Algoverse\lme_v2_work\LongMemEval-V2\evaluation\run_eval.py --method rag_query_to_slice_notes --data-root D:\Projects\Algoverse\lme_v2_work\data\longmemeval-v2 --domain web --tier small --output-dir D:\Projects\Algoverse\lme_v2_work\runs\quickcheck\rag_query_to_slice_notes_web_small --reader-model Qwen/Qwen3.5-9B --reader-base-url https://openrouter.ai/api/v1 --reader-api-key-env OPENROUTER_API_KEY --controller-model Qwen/Qwen3.5-9B --controller-base-url https://openrouter.ai/api/v1 --controller-api-key-env OPENROUTER_API_KEY --embedding-model Qwen/Qwen3-Embedding-8B --embedding-base-url https://openrouter.ai/api/v1 --embedding-api-key-env OPENROUTER_API_KEY --evaluator-model gpt-5.2 --evaluator-api-key-env OPENAI_API_KEY --evaluator-reasoning-effort medium --limit 5


run_eval.py exited 1 on attempt 1/2; retrying...


Running: D:\Projects\Algoverse\.venv\Scripts\python.exe D:\Projects\Algoverse\lme_v2_work\LongMemEval-V2\evaluation\run_eval.py --method rag_query_to_slice_notes --data-root D:\Projects\Algoverse\lme_v2_work\data\longmemeval-v2 --domain enterprise --tier small --output-dir D:\Projects\Algoverse\lme_v2_work\runs\quickcheck\rag_query_to_slice_notes_enterprise_small --reader-model Qwen/Qwen3.5-9B --reader-base-url https://openrouter.ai/api/v1 --reader-api-key-env OPENROUTER_API_KEY --controller-model Qwen/Qwen3.5-9B --controller-base-url https://openrouter.ai/api/v1 --controller-api-key-env OPENROUTER_API_KEY --embedding-model Qwen/Qwen3-Embedding-8B --embedding-base-url https://openrouter.ai/api/v1 --embedding-api-key-env OPENROUTER_API_KEY --evaluator-model gpt-5.2 --evaluator-api-key-env OPENAI_API_KEY --evaluator-reasoning-effort medium --limit 5


run_eval.py exited 1 on attempt 1/2; retrying...


Quick check complete -- inspect the output above for errors before running the full arm below.


In [10]:
slice_notes_dirs = run_lme_both_domains("rag_query_to_slice_notes", subprocess_timeout=10800, retries=2)


Running: D:\Projects\Algoverse\.venv\Scripts\python.exe D:\Projects\Algoverse\lme_v2_work\LongMemEval-V2\evaluation\run_eval.py --method rag_query_to_slice_notes --data-root D:\Projects\Algoverse\lme_v2_work\data\longmemeval-v2 --domain web --tier small --output-dir D:\Projects\Algoverse\lme_v2_work\runs\rag_query_to_slice_notes_web_small --reader-model Qwen/Qwen3.5-9B --reader-base-url https://openrouter.ai/api/v1 --reader-api-key-env OPENROUTER_API_KEY --controller-model Qwen/Qwen3.5-9B --controller-base-url https://openrouter.ai/api/v1 --controller-api-key-env OPENROUTER_API_KEY --embedding-model Qwen/Qwen3-Embedding-8B --embedding-base-url https://openrouter.ai/api/v1 --embedding-api-key-env OPENROUTER_API_KEY --evaluator-model gpt-5.2 --evaluator-api-key-env OPENAI_API_KEY --evaluator-reasoning-effort medium


Running: D:\Projects\Algoverse\.venv\Scripts\python.exe D:\Projects\Algoverse\lme_v2_work\LongMemEval-V2\evaluation\run_eval.py --method rag_query_to_slice_notes --data-root D:\Projects\Algoverse\lme_v2_work\data\longmemeval-v2 --domain enterprise --tier small --output-dir D:\Projects\Algoverse\lme_v2_work\runs\rag_query_to_slice_notes_enterprise_small --reader-model Qwen/Qwen3.5-9B --reader-base-url https://openrouter.ai/api/v1 --reader-api-key-env OPENROUTER_API_KEY --controller-model Qwen/Qwen3.5-9B --controller-base-url https://openrouter.ai/api/v1 --controller-api-key-env OPENROUTER_API_KEY --embedding-model Qwen/Qwen3-Embedding-8B --embedding-base-url https://openrouter.ai/api/v1 --embedding-api-key-env OPENROUTER_API_KEY --evaluator-model gpt-5.2 --evaluator-api-key-env OPENAI_API_KEY --evaluator-reasoning-effort medium


## 10. Gate 3 -- combine domains and compare against Table 2


In [11]:
import pandas as pd

PAPER_SMALL = {
    "no_retrieval": 0.013,
    "rag_query_to_slice": 0.428,
    "rag_query_to_slice_notes": 0.510,
}

rows = []
combined_by_method = {}
for method in ARMS:
    combined = combine_domains(method)
    combined_by_method[method] = combined

    records = load_per_question(method, "web") + load_per_question(method, "enterprise")
    n = len(records)
    k = sum(1 for r in records if r["score_bool"])
    p, lo, hi = wilson_interval(k, n)

    rows.append({
        "method": method,
        "paper_small": PAPER_SMALL[method],
        "ours": round(p, 4) if p is not None else None,
        "delta": round(p - PAPER_SMALL[method], 4) if p is not None else None,
        "95pct_ci": f"[{lo:.3f}, {hi:.3f}]" if lo is not None else None,
        "n": n,
        "k_correct": k,
    })

comparison_df = pd.DataFrame(rows).set_index("method")
comparison_df


,paper_small,ours,delta,95pct_ci,n,k_correct
method,,,,,,
no_retrieval,0.013,0.0288,0.0158,"[0.017, 0.049]",451,13
rag_query_to_slice,0.428,0.4213,-0.0067,"[0.377, 0.467]",451,190
rag_query_to_slice_notes,0.510,0.4656,-0.0444,"[0.420, 0.512]",451,210


In [12]:
scores = {row["method"]: row["ours"] for row in rows}
ordering_holds = scores["no_retrieval"] < scores["rag_query_to_slice"] < scores["rag_query_to_slice_notes"]
notes_gap = scores["rag_query_to_slice_notes"] - scores["rag_query_to_slice"]

print(f"Ordering (no_retrieval < slice < slice+notes) holds: {ordering_holds}")
print(f"Notes gap: {notes_gap:+.4f} (paper: +0.082)")
print(
    "\nThe paper reports single runs at temperature 0.6 with a sampling 9B reader, so "
    "run-to-run variance around any one point estimate is expected and uncharacterized "
    "in the paper itself. The ordering above and the size of the notes gap are more "
    "robust replication signals than any single absolute score landing exactly on target -- "
    "see Section 13 to convert these point estimates into a variance estimate."
)


Ordering (no_retrieval < slice < slice+notes) holds: True
Notes gap: +0.0443 (paper: +0.082)

The paper reports single runs at temperature 0.6 with a sampling 9B reader, so run-to-run variance around any one point estimate is expected and uncharacterized in the paper itself. The ordering above and the size of the notes gap are more robust replication signals than any single absolute score landing exactly on target -- see Section 13 to convert these point estimates into a variance estimate.


## 11. Per-ability breakdown

`static` / `dynamic` / `procedure` (workflow knowledge) / `gotchas` are the harness's non-abstention
categories; `combined_abstention_by_category` merges each `-abs` (premise-awareness) variant back into its
parent category, matching how the paper's five-ability breakdown collapses into these four reported groups
plus an abstention-overall figure.


In [13]:
category_rows = []
for method in ARMS:
    combined = combined_by_method[method]
    for cat, stats in combined["non_abstention_by_category"].items():
        category_rows.append({"method": method, "category": cat, **stats})
    category_rows.append({"method": method, "category": "abstention_overall", **combined["abstention_overall"]})

category_df = pd.DataFrame(category_rows).set_index(["method", "category"])
category_df


count  pct_correct  \
method                   category                                 
no_retrieval             static                134     0.029851   
                         dynamic                86     0.000000   
                         procedure              74     0.040541   
                         gotchas                29     0.172414   
                         abstention_overall    128     0.007812   
rag_query_to_slice       static                134     0.537313   
                         dynamic                86     0.546512   
                         procedure              74     0.567568   
                         gotchas                29     0.172414   
                         abstention_overall    128     0.187500   
rag_query_to_slice_notes static                134     0.559701   
                         dynamic                86     0.616279   
                         procedure              74     0.662162   
                         gotchas                29     0.241379   
                         abstention_overall    128     0.203125   

                                             pct_answered_wrong  pct_unknown  
method                   category                                             
no_retrieval             static                        0.029851     0.940299  
                         dynamic                       0.116279     0.883721  
                         procedure                     0.054054     0.905405  
                         gotchas                       0.827586     0.000000  
                         abstention_overall            0.039062     0.953125  
rag_query_to_slice       static                        0.201493     0.261194  
                         dynamic                       0.209302     0.244186  
                         procedure                     0.337838     0.094595  
                         gotchas                       0.724138     0.103448  
                         abstention_overall            0.484375     0.328125  
rag_query_to_slice_notes static                        0.291045     0.149254  
                         dynamic                       0.279070     0.104651  
                         procedure                     0.337838     0.000000  
                         gotchas                       0.758621     0.000000  
                         abstention_overall            0.546875     0.250000

## 12. Cost report

Reader token usage is recorded per run by the harness; embedding and judge token usage aren't broken out in
`aggregated_metrics.json`, so check your provider dashboards for those (they're the smaller line items --
see the plan file's cost table).


In [14]:
# Update these to match whatever hosted pricing you're actually using (README's OpenRouter figures).
READER_PRICE_PER_M_IN = 0.10
READER_PRICE_PER_M_OUT = 0.15

cost_rows = []
for method in ARMS:
    tokens = combined_by_method[method]["tokens"]
    prompt_tok = tokens.get("prompt_tokens") or 0
    completion_tok = tokens.get("completion_tokens") or 0
    cost = (prompt_tok / 1e6) * READER_PRICE_PER_M_IN + (completion_tok / 1e6) * READER_PRICE_PER_M_OUT
    cost_rows.append({
        "method": method,
        "prompt_tokens": prompt_tok,
        "completion_tokens": completion_tok,
        "reader_cost_usd": round(cost, 4),
    })

cost_df = pd.DataFrame(cost_rows).set_index("method")
print(cost_df)
print(f"\nTotal reader cost (this notebook's 3 arms): ${cost_df['reader_cost_usd'].sum():.2f}")
print("Add embedding + note-generation + judge spend from your provider dashboards for the full total.")


                          prompt_tokens  completion_tokens  reader_cost_usd
method                                                                     
no_retrieval                     118807            3221623           0.4951
rag_query_to_slice             40703869            2550144           4.4529
rag_query_to_slice_notes       41204276            2591413           4.5091

Total reader cost (this notebook's 3 arms): $9.46
Add embedding + note-generation + judge spend from your provider dashboards for the full total.


## 13. Appendix: seed variance (optional, Gate 4)

Off by default -- this roughly doubles the RAG arms' cost for two extra `--shuffle-questions-seed` runs.
Turn on `RUN_SEED_VARIANCE` if the Gate 3 comparison above needs a real variance estimate rather than a
qualitative read of the ordering and the notes gap.


In [15]:
RUN_SEED_VARIANCE = False
EXTRA_SEEDS = [1337, 2026]

if RUN_SEED_VARIANCE:
    seed_scores: dict[str, list[float]] = {method: [scores[method]] for method in ARMS}
    for seed in EXTRA_SEEDS:
        seed_output_root = OUTPUT_ROOT / f"seed_{seed}"
        for method in ARMS:
            run_lme_both_domains(
                method,
                shuffle_seed=seed,
                output_root=seed_output_root,
                subprocess_timeout=10800,
                retries=2,
            )
            seed_records = (
                load_per_question(method, "web", output_root=seed_output_root)
                + load_per_question(method, "enterprise", output_root=seed_output_root)
            )
            seed_scores[method].append(sum(r["score_bool"] for r in seed_records) / len(seed_records))

    for method, values in seed_scores.items():
        mean = sum(values) / len(values)
        spread = max(values) - min(values)
        print(f"{method}: seeds={[round(v, 4) for v in values]} mean={mean:.4f} range={spread:.4f}")
else:
    print("RUN_SEED_VARIANCE is False -- skipped. Flip it to True to spend the extra ~$20 on 2 more seeds.")


RUN_SEED_VARIANCE is False -- skipped. Flip it to True to spend the extra ~$20 on 2 more seeds.


## 14. Appendix: what's out of scope and why

- **`codex` baseline** -- an off-the-shelf Codex-agent baseline, ~$0.30/question at the paper's
  3-concurrent-process cap (~$135 for 451 questions). Excluded from this notebook's scope by design; see
  the plan file's cost table if you want to add it back.
- **`agentrunbook_r` / `agentrunbook_c`** -- the paper's own proposed methods, not baselines. Excluded
  because this notebook's stated goal is baseline reproduction specifically.
- **Medium tier** -- `evaluation/run_eval.py` hardcodes `trajectory_pool_root: None` with no CLI override,
  so a naive Medium run reinserts trajectories per-question (451 x ~498) instead of reusing the
  content-addressed artifact pool the harness supports for this exact purpose. Reaching Medium means
  writing a custom memory-config JSON with the pool root set and invoking `evaluation/harness.py` directly
  -- a separate effort with its own feasibility spike, not attempted here.
